In [0]:
# Criando os esquemas para a arquitetura Medalhão dentro do catálogo workspace
spark.sql("CREATE SCHEMA IF NOT EXISTS workspace.bronze")
spark.sql("CREATE SCHEMA IF NOT EXISTS workspace.silver")
spark.sql("CREATE SCHEMA IF NOT EXISTS workspace.gold")

print("Esquemas bronze, silver e gold criados com sucesso!")

Esquemas bronze, silver e gold criados com sucesso!


In [0]:
from pyspark.sql.functions import current_timestamp, lit

volume_path = "/Volumes/workspace/default/raw_files"

datasets = {
    "orders": "olist_orders_dataset.csv",
    "order_items": "olist_order_items_dataset.csv",
    "customers": "olist_customers_dataset.csv",
    "products": "olist_products_dataset.csv",
    "payments": "olist_order_payments_dataset.csv",
    "reviews": "olist_order_reviews_dataset.csv",
    "sellers": "olist_sellers_dataset.csv",
    "category_translation": "product_category_name_translation.csv"
}

for table_name, file_name in datasets.items():
    file_full_path = f"{volume_path}/{file_name}"
    print(f"Ingerindo {file_name} -> workspace.bronze.{table_name}...")
    
    df_raw = spark.read.format("csv") \
        .option("header", "true") \
        .option("inferSchema", "true") \
        .load(file_full_path)
    
    # Adicionando metadados de auditoria e rastreabilidade (requisito de engenharia)
    df_bronze = df_raw \
        .withColumn("_data_ingestao", current_timestamp()) \
        .withColumn("_origem_arquivo", lit(file_full_path))
    
    df_bronze.write.format("delta") \
        .mode("overwrite") \
        .saveAsTable(f"workspace.bronze.{table_name}")

print("Todas as tabelas Bronze foram persistidas com sucesso!")

Ingerindo olist_orders_dataset.csv -> workspace.bronze.orders...
Ingerindo olist_order_items_dataset.csv -> workspace.bronze.order_items...
Ingerindo olist_customers_dataset.csv -> workspace.bronze.customers...
Ingerindo olist_products_dataset.csv -> workspace.bronze.products...
Ingerindo olist_order_payments_dataset.csv -> workspace.bronze.payments...
Ingerindo olist_order_reviews_dataset.csv -> workspace.bronze.reviews...
Ingerindo olist_sellers_dataset.csv -> workspace.bronze.sellers...
Ingerindo product_category_name_translation.csv -> workspace.bronze.category_translation...
Todas as tabelas Bronze foram persistidas com sucesso!


In [0]:
from pyspark.sql.functions import col, to_timestamp, trim

# Pedidos
orders_silver = (
    spark.table("workspace.bronze.orders")
    .dropDuplicates(["order_id"])
    .withColumn("order_purchase_timestamp",
                to_timestamp("order_purchase_timestamp"))
    .withColumn("order_approved_at",
                to_timestamp("order_approved_at"))
    .withColumn("order_delivered_carrier_date",
                to_timestamp("order_delivered_carrier_date"))
    .withColumn("order_delivered_customer_date",
                to_timestamp("order_delivered_customer_date"))
    .withColumn("order_estimated_delivery_date",
                to_timestamp("order_estimated_delivery_date"))
    .withColumn("order_status", trim(col("order_status")))
)

orders_silver.write.format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("workspace.silver.orders")


# Itens dos pedidos
items_silver = (
    spark.table("workspace.bronze.order_items")
    .dropDuplicates(["order_id", "order_item_id"])
    .withColumn("shipping_limit_date",
                to_timestamp("shipping_limit_date"))
)

items_silver.write.format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("workspace.silver.order_items")


# Clientes
customers_silver = (
    spark.table("workspace.bronze.customers")
    .dropDuplicates(["customer_id"])
    .withColumn("customer_state", trim(col("customer_state")))
)

customers_silver.write.format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("workspace.silver.customers")


# Produtos
products_silver = (
    spark.table("workspace.bronze.products")
    .dropDuplicates(["product_id"])
)

products_silver.write.format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("workspace.silver.products")


# Pagamentos
payments_silver = (
    spark.table("workspace.bronze.payments")
    .dropDuplicates(["order_id", "payment_sequential"])
    .withColumn("payment_type", trim(col("payment_type")))
)

payments_silver.write.format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("workspace.silver.payments")


# Avaliações
reviews_silver = (
    spark.table("workspace.bronze.reviews")
    .dropDuplicates(["review_id"])
    .withColumn("review_comment_title",
                trim(col("review_comment_title")))
    .withColumn("review_comment_message",
                trim(col("review_comment_message")))
)

reviews_silver.write.format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("workspace.silver.reviews")


# Vendedores
sellers_silver = (
    spark.table("workspace.bronze.sellers")
    .dropDuplicates(["seller_id"])
    .withColumn("seller_state", trim(col("seller_state")))
)

sellers_silver.write.format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("workspace.silver.sellers")


# Tradução das categorias
category_silver = (
    spark.table("workspace.bronze.category_translation")
    .dropDuplicates(["product_category_name"])
)

category_silver.write.format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("workspace.silver.category_translation")

print("Camada Silver criada com sucesso!")

Camada Silver criada com sucesso!


In [0]:
spark.sql("SHOW TABLES IN workspace.silver").show()

+--------+--------------------+-----------+
|database|           tableName|isTemporary|
+--------+--------------------+-----------+
|  silver|category_translation|      false|
|  silver|           customers|      false|
|  silver|         order_items|      false|
|  silver|              orders|      false|
|  silver|            payments|      false|
|  silver|            products|      false|
|  silver|             reviews|      false|
|  silver|             sellers|      false|
+--------+--------------------+-----------+



In [0]:
spark.sql("""
SELECT
    order_status,
    COUNT(*) AS quantidade_pedidos
FROM workspace.silver.orders
GROUP BY order_status
ORDER BY quantidade_pedidos DESC
""").show()

+------------+------------------+
|order_status|quantidade_pedidos|
+------------+------------------+
|   delivered|             96478|
|     shipped|              1107|
|    canceled|               625|
| unavailable|               609|
|    invoiced|               314|
|  processing|               301|
|     created|                 5|
|    approved|                 2|
+------------+------------------+



In [0]:
from pyspark.sql.functions import col, count, when, isnan

# Verificação de nulos nas principais colunas de pedidos
orders_quality = spark.sql("""
SELECT
    COUNT(*) AS total_registros,
    SUM(CASE WHEN order_id IS NULL THEN 1 ELSE 0 END) AS nulos_order_id,
    SUM(CASE WHEN customer_id IS NULL THEN 1 ELSE 0 END) AS nulos_customer_id,
    SUM(CASE WHEN order_status IS NULL THEN 1 ELSE 0 END) AS nulos_order_status,
    SUM(CASE WHEN order_purchase_timestamp IS NULL THEN 1 ELSE 0 END) AS nulos_data_compra,
    SUM(CASE WHEN order_delivered_customer_date IS NULL THEN 1 ELSE 0 END) AS nulos_data_entrega,
    SUM(CASE WHEN order_estimated_delivery_date IS NULL THEN 1 ELSE 0 END) AS nulos_data_estimada
FROM workspace.silver.orders
""")

orders_quality.show()

+---------------+--------------+-----------------+------------------+-----------------+------------------+-------------------+
|total_registros|nulos_order_id|nulos_customer_id|nulos_order_status|nulos_data_compra|nulos_data_entrega|nulos_data_estimada|
+---------------+--------------+-----------------+------------------+-----------------+------------------+-------------------+
|          99441|             0|                0|                 0|                0|              2965|                  0|
+---------------+--------------+-----------------+------------------+-----------------+------------------+-------------------+



In [0]:
spark.sql("""
SELECT
    MIN(review_score) AS menor_nota,
    MAX(review_score) AS maior_nota,
    COUNT(*) AS total_avaliacoes,
    SUM(CASE WHEN review_score IS NULL THEN 1 ELSE 0 END) AS notas_nulas
FROM workspace.silver.reviews
""").show()

+----------+--------------------+----------------+-----------+
|menor_nota|          maior_nota|total_avaliacoes|notas_nulas|
+----------+--------------------+----------------+-----------+
|         "|seria mais coeren...|          102958|       2315|
+----------+--------------------+----------------+-----------+



In [0]:
spark.sql("""
SELECT
    COUNT(*) AS total_itens,
    SUM(CASE WHEN price < 0 THEN 1 ELSE 0 END) AS precos_negativos,
    SUM(CASE WHEN freight_value < 0 THEN 1 ELSE 0 END) AS fretes_negativos,
    MIN(price) AS menor_preco,
    MAX(price) AS maior_preco
FROM workspace.silver.order_items
""").show()

+-----------+----------------+----------------+-----------+-----------+
|total_itens|precos_negativos|fretes_negativos|menor_preco|maior_preco|
+-----------+----------------+----------------+-----------+-----------+
|     112650|               0|               0|       0.85|     6735.0|
+-----------+----------------+----------------+-----------+-----------+



In [0]:
spark.table("workspace.silver.order_items").printSchema()

spark.sql("""
SELECT
    price,
    freight_value,
    shipping_limit_date
FROM workspace.silver.order_items
LIMIT 10
""").show(truncate=False)

root
 |-- order_id: string (nullable = true)
 |-- order_item_id: integer (nullable = true)
 |-- product_id: string (nullable = true)
 |-- seller_id: string (nullable = true)
 |-- shipping_limit_date: timestamp (nullable = true)
 |-- price: double (nullable = true)
 |-- freight_value: double (nullable = true)
 |-- _data_ingestao: timestamp (nullable = true)
 |-- _origem_arquivo: string (nullable = true)

+------+-------------+-------------------+
|price |freight_value|shipping_limit_date|
+------+-------------+-------------------+
|48.9  |16.6         |2017-09-15 18:04:37|
|29.99 |15.31        |2018-08-21 03:35:17|
|159.0 |27.19        |2018-03-15 19:28:51|
|179.99|85.97        |2017-10-03 22:46:14|
|49.99 |18.12        |2017-10-24 12:28:16|
|96.99 |16.44        |2017-10-05 02:24:43|
|39.85 |12.79        |2018-03-28 23:07:23|
|55.0  |7.65         |2018-08-06 21:50:11|
|133.0 |16.69        |2017-05-19 17:35:15|
|18.0  |7.87         |2018-06-29 03:31:55|
+------+-------------+------------

In [0]:
from pyspark.sql.functions import col, coalesce, lit, to_date, year, month, dayofmonth, dayofweek, date_format

# 1. Dimensão Cliente
dim_cliente = spark.sql("""
    SELECT DISTINCT
        customer_id,
        customer_unique_id,
        customer_city,
        customer_state
    FROM workspace.silver.customers
""")

dim_cliente.write.format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("workspace.gold.dim_cliente")

# 2. Dimensão Vendedor
dim_vendedor = spark.sql("""
    SELECT DISTINCT
        seller_id,
        seller_city,
        seller_state
    FROM workspace.silver.sellers
""")

dim_vendedor.write.format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("workspace.gold.dim_vendedor")

# 3. Dimensão Produto (com tradução da categoria)
dim_produto = spark.sql("""
    SELECT
        p.product_id,
        COALESCE(t.product_category_name_english, p.product_category_name, 'nao_informado') AS categoria_produto,
        p.product_weight_g AS peso_gramas,
        p.product_length_cm AS comprimento_cm,
        p.product_height_cm AS altura_cm,
        p.product_width_cm AS largura_cm
    FROM workspace.silver.products p
    LEFT JOIN workspace.silver.category_translation t
        ON p.product_category_name = t.product_category_name
""")

dim_produto.write.format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("workspace.gold.dim_produto")

# 4. Dimensão Tempo / Calendário
dim_tempo = spark.sql("""
    SELECT DISTINCT
        TO_DATE(order_purchase_timestamp) AS data,
        YEAR(order_purchase_timestamp) AS ano,
        MONTH(order_purchase_timestamp) AS mes,
        DAY(order_purchase_timestamp) AS dia,
        DAYOFWEEK(order_purchase_timestamp) AS dia_semana,
        DATE_FORMAT(order_purchase_timestamp, 'MMMM') AS nome_mes
    FROM workspace.silver.orders
    WHERE order_purchase_timestamp IS NOT NULL
""")

dim_tempo.write.format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("workspace.gold.dim_tempo")

print("Tabelas Dimensão criadas com sucesso na camada Gold!")

Tabelas Dimensão criadas com sucesso na camada Gold!


In [0]:
fato_itens = spark.sql("""
    SELECT
        i.order_id,
        i.order_item_id,
        i.product_id,
        i.seller_id,
        o.customer_id,
        TO_DATE(o.order_purchase_timestamp) AS data_compra,
        o.order_status,
        
        -- Métricas financeiras com casting seguro
        TRY_CAST(i.price AS DECIMAL(10,2)) AS valor_item,
        TRY_CAST(i.freight_value AS DECIMAL(10,2)) AS valor_frete,
        CAST(TRY_CAST(i.price AS DECIMAL(10,2)) + TRY_CAST(i.freight_value AS DECIMAL(10,2)) AS DECIMAL(10,2)) AS valor_total_item,
        
        -- Métricas temporais e logísticas
        o.order_purchase_timestamp,
        o.order_delivered_customer_date,
        o.order_estimated_delivery_date,
        DATEDIFF(TO_DATE(o.order_delivered_customer_date), TO_DATE(o.order_purchase_timestamp)) AS dias_entrega_real,
        DATEDIFF(TO_DATE(o.order_estimated_delivery_date), TO_DATE(o.order_purchase_timestamp)) AS dias_entrega_estimada,
        CASE 
            WHEN o.order_delivered_customer_date IS NOT NULL 
                 AND o.order_estimated_delivery_date IS NOT NULL 
                 AND o.order_delivered_customer_date > o.order_estimated_delivery_date 
            THEN 1 
            ELSE 0 
        END AS flag_atraso_entrega,
        
        -- Métrica de satisfação com TRY_CAST defensivo no AVG
        r.review_score
    FROM workspace.silver.order_items i
    INNER JOIN workspace.silver.orders o 
        ON i.order_id = o.order_id
    LEFT JOIN (
        SELECT 
            order_id, 
            AVG(TRY_CAST(review_score AS DOUBLE)) AS review_score
        FROM workspace.silver.reviews
        GROUP BY order_id
    ) r ON i.order_id = r.order_id
""")

fato_itens.write.format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("workspace.gold.fato_itens_pedidos")

print("Tabela Fato criada com sucesso na camada Gold!")

Tabela Fato criada com sucesso na camada Gold!


In [0]:
fato_pagamentos = spark.sql("""
    SELECT
        p.order_id,
        p.payment_sequential,
        p.payment_type,
        p.payment_installments,
        CAST(p.payment_value AS DECIMAL(10,2)) AS valor_pagamento,
        TO_DATE(o.order_purchase_timestamp) AS data_compra
    FROM workspace.silver.payments p
    INNER JOIN workspace.silver.orders o
        ON p.order_id = o.order_id
""")

fato_pagamentos.write.format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("workspace.gold.fato_pagamentos")

print("Tabela de Pagamentos criada com sucesso na camada Gold!")

Tabela de Pagamentos criada com sucesso na camada Gold!


In [0]:
spark.sql("SHOW TABLES IN workspace.gold").show()

+--------+------------------+-----------+
|database|         tableName|isTemporary|
+--------+------------------+-----------+
|    gold|       dim_cliente|      false|
|    gold|       dim_produto|      false|
|    gold|         dim_tempo|      false|
|    gold|      dim_vendedor|      false|
|    gold|fato_itens_pedidos|      false|
|    gold|   fato_pagamentos|      false|
+--------+------------------+-----------+



In [0]:
spark.sql("""
SELECT
    COUNT(*) AS total_itens,
    SUM(CASE WHEN valor_item IS NULL THEN 1 ELSE 0 END) AS itens_sem_valor,
    SUM(CASE WHEN valor_frete IS NULL THEN 1 ELSE 0 END) AS itens_sem_frete,
    SUM(CASE WHEN data_compra IS NULL THEN 1 ELSE 0 END) AS datas_compra_nulas,
    SUM(flag_atraso_entrega) AS itens_atrasados
FROM workspace.gold.fato_itens_pedidos
""").show()

+-----------+---------------+---------------+------------------+---------------+
|total_itens|itens_sem_valor|itens_sem_frete|datas_compra_nulas|itens_atrasados|
+-----------+---------------+---------------+------------------+---------------+
|     112650|              0|              0|                 0|           8715|
+-----------+---------------+---------------+------------------+---------------+



In [0]:
spark.sql("""
SELECT
    payment_type,
    COUNT(*) AS quantidade,
    ROUND(SUM(valor_pagamento), 2) AS valor_total
FROM workspace.gold.fato_pagamentos
GROUP BY payment_type
ORDER BY valor_total DESC
""").show()

+------------+----------+-----------+
|payment_type|quantidade|valor_total|
+------------+----------+-----------+
| credit_card|     76795|12542084.19|
|      boleto|     19784| 2869361.27|
|     voucher|      5775|  379436.87|
|  debit_card|      1529|  217989.79|
| not_defined|         3|       0.00|
+------------+----------+-----------+



Pergunta 1: Desempenho Comercial por Categoria

Quais são as 10 categorias de produtos que geram maior volume de receita bruta e qual o ticket médio de cada uma?

In [0]:
pergunta_1 = spark.sql("""
    SELECT 
        p.categoria_produto,
        COUNT(f.order_id) AS total_itens_vendidos,
        ROUND(SUM(f.valor_item), 2) AS receita_bruta_produtos,
        ROUND(AVG(f.valor_item), 2) AS ticket_medio_item
    FROM workspace.gold.fato_itens_pedidos f
    INNER JOIN workspace.gold.dim_produto p
        ON f.product_id = p.product_id
    GROUP BY p.categoria_produto
    ORDER BY receita_bruta_produtos DESC
    LIMIT 10
""")

pergunta_1.show(truncate=False)

+---------------------+--------------------+----------------------+-----------------+
|categoria_produto    |total_itens_vendidos|receita_bruta_produtos|ticket_medio_item|
+---------------------+--------------------+----------------------+-----------------+
|health_beauty        |9670                |1258681.34            |130.16           |
|watches_gifts        |5991                |1205005.68            |201.14           |
|bed_bath_table       |11115               |1036988.68            |93.30            |
|sports_leisure       |8641                |988048.97             |114.34           |
|computers_accessories|7827                |911954.32             |116.51           |
|furniture_decor      |8334                |729762.49             |87.56            |
|cool_stuff           |3796                |635290.85             |167.36           |
|housewares           |6964                |632248.66             |90.79            |
|auto                 |4235                |592720.11 

Pergunta 2: Logística e Prazos de Entrega por Estado

Qual é o tempo médio real de entrega (em dias) por UF de destino e qual a taxa percentual de pedidos com atraso?

In [0]:
pergunta_2 = spark.sql("""
    SELECT 
        c.customer_state AS estado_destino,
        COUNT(f.order_id) AS total_entregas,
        ROUND(AVG(f.dias_entrega_real), 1) AS media_dias_entrega,
        ROUND(AVG(f.dias_entrega_estimada), 1) AS media_dias_estimados,
        ROUND((SUM(f.flag_atraso_entrega) * 100.0 / COUNT(f.order_id)), 2) AS taxa_atraso_percentual
    FROM workspace.gold.fato_itens_pedidos f
    INNER JOIN workspace.gold.dim_cliente c
        ON f.customer_id = c.customer_id
    WHERE f.order_status = 'delivered'
      AND f.dias_entrega_real IS NOT NULL
    GROUP BY c.customer_state
    ORDER BY taxa_atraso_percentual DESC
""")

pergunta_2.show(27, truncate=False)

+--------------+--------------+------------------+--------------------+----------------------+
|estado_destino|total_entregas|media_dias_entrega|media_dias_estimados|taxa_atraso_percentual|
+--------------+--------------+------------------+--------------------+----------------------+
|AL            |427           |24.4              |33.2                |24.12                 |
|MA            |800           |21.6              |31.5                |20.38                 |
|SE            |375           |21.4              |31.4                |16.27                 |
|PI            |523           |19.3              |30.8                |15.49                 |
|CE            |1426          |20.9              |32.0                |15.29                 |
|BA            |3683          |19.2              |30.2                |13.68                 |
|RJ            |14143         |15.1              |27.1                |12.97                 |
|PA            |1054          |23.7              |

Pergunta 3: Impacto Logístico na Satisfação (Review Score)

Qual é a nota média de avaliação atribuída pelo cliente quando a entrega atrasa versus quando é entregue no prazo?

In [0]:
pergunta_3 = spark.sql("""
    SELECT 
        CASE 
            WHEN flag_atraso_entrega = 1 THEN 'Com Atraso'
            ELSE 'No Prazo'
        END AS situacao_entrega,
        COUNT(order_id) AS total_itens,
        ROUND(AVG(review_score), 2) AS nota_media_avaliacao,
        ROUND(SUM(CASE WHEN review_score = 1 THEN 1 ELSE 0 END) * 100.0 / COUNT(order_id), 2) AS percentual_nota_1,
        ROUND(SUM(CASE WHEN review_score = 5 THEN 1 ELSE 0 END) * 100.0 / COUNT(order_id), 2) AS percentual_nota_5
    FROM workspace.gold.fato_itens_pedidos
    WHERE order_status = 'delivered'
      AND review_score IS NOT NULL
    GROUP BY flag_atraso_entrega
""")

pergunta_3.show(truncate=False)

+----------------+-----------+--------------------+-----------------+-----------------+
|situacao_entrega|total_itens|nota_media_avaliacao|percentual_nota_1|percentual_nota_5|
+----------------+-----------+--------------------+-----------------+-----------------+
|No Prazo        |100375     |4.21                |8.35             |60.56            |
|Com Atraso      |8462       |2.55                |46.87            |22.17            |
+----------------+-----------+--------------------+-----------------+-----------------+



Pergunta 4: Meios de Pagamento e Concentração de Parcelas

Qual a distribuição de faturamento por método de pagamento e qual o número médio de parcelas no cartão de crédito?

In [0]:
pergunta_4 = spark.sql("""
    SELECT 
        payment_type AS metodo_pagamento,
        COUNT(order_id) AS total_transacoes,
        ROUND(SUM(valor_pagamento), 2) AS faturamento_total,
        ROUND(AVG(valor_pagamento), 2) AS ticket_medio_transacao,
        ROUND(AVG(payment_installments), 1) AS media_parcelas
    FROM workspace.gold.fato_pagamentos
    GROUP BY payment_type
    ORDER BY faturamento_total DESC
""")

pergunta_4.show(truncate=False)

+----------------+----------------+-----------------+----------------------+--------------+
|metodo_pagamento|total_transacoes|faturamento_total|ticket_medio_transacao|media_parcelas|
+----------------+----------------+-----------------+----------------------+--------------+
|credit_card     |76795           |12542084.19      |163.32                |3.5           |
|boleto          |19784           |2869361.27       |145.03                |1.0           |
|voucher         |5775            |379436.87        |65.70                 |1.0           |
|debit_card      |1529            |217989.79        |142.57                |1.0           |
|not_defined     |3               |0.00             |0.00                  |1.0           |
+----------------+----------------+-----------------+----------------------+--------------+



Pergunta 5: Origem vs. Destino e Impacto no Custo do Frete

Qual o custo médio de frete e o volume de transações para entregas no mesmo estado (intraestadual) versus entregas entre estados diferentes (interestadual)?

In [0]:
pergunta_5 = spark.sql("""
    SELECT 
        CASE 
            WHEN v.seller_state = c.customer_state THEN 'Intraestadual (Mesmo Estado)'
            ELSE 'Interestadual (Estados Diferentes)'
        END AS tipo_fluxo_logistico,
        COUNT(f.order_id) AS total_itens,
        ROUND(AVG(f.valor_frete), 2) AS frete_medio,
        ROUND(AVG(f.dias_entrega_real), 1) AS media_dias_entrega,
        ROUND((SUM(f.flag_atraso_entrega) * 100.0 / COUNT(f.order_id)), 2) AS taxa_atraso_percentual
    FROM workspace.gold.fato_itens_pedidos f
    INNER JOIN workspace.gold.dim_vendedor v ON f.seller_id = v.seller_id
    INNER JOIN workspace.gold.dim_cliente c ON f.customer_id = c.customer_id
    WHERE f.order_status = 'delivered'
      AND f.dias_entrega_real IS NOT NULL
    GROUP BY 
        CASE 
            WHEN v.seller_state = c.customer_state THEN 'Intraestadual (Mesmo Estado)'
            ELSE 'Interestadual (Estados Diferentes)'
        END
""")

pergunta_5.show(truncate=False)

+----------------------------------+-----------+-----------+------------------+----------------------+
|tipo_fluxo_logistico              |total_itens|frete_medio|media_dias_entrega|taxa_atraso_percentual|
+----------------------------------+-----------+-----------+------------------+----------------------+
|Interestadual (Estados Diferentes)|70328      |23.63      |15.0              |9.00                  |
|Intraestadual (Mesmo Estado)      |39861      |13.45      |7.9               |5.98                  |
+----------------------------------+-----------+-----------+------------------+----------------------+

